# Corn Strategy Backtest Research

This is a corn-only sprint notebook. It keeps the work intentionally narrow: start with the requested generic Signal A/B checks, test whether a simple Momentum/MR benchmark survives walk-forward, then use the Cargill data as a small physical-risk filter rather than forcing it to be the whole alpha.

The final recommendation is not a broad fitted model. It is a risk-controlled Momentum/MR corn sleeve with a Cargill disagreement filter.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from research_config import CORN_TRAIN_END, SPLIT_DATE
from grain_research import (
    load_train_set,
    build_product_flow_feature_panels,
    build_corn_product_flow_signal_universe,
    corn_signal_set_families,
    corn_average_all_signals,
    corn_equal_family_signal,
    corn_select_by_ic_signal,
    corn_trend_mr_family_signal,
    mean_product_flow_signals,
    corn_positions_from_signal,
    backtest_positions_product_flow,
    summarize_corn_backtest,
    clean_product_flow_signal,
    product_flow_performance_metrics,
    corn_abundant_supply_masks,
    make_corn_candidate,
    run_corn_supply_guard_tests,
    product_flow_period_performance,
)


pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA_DIR = "train_set"
COMMODITY = "CORN"
TRAIN_END = pd.Timestamp(CORN_TRAIN_END)
OOS_START = pd.Timestamp(SPLIT_DATE)
DD_CAPITAL_USD = 10_000.0

DD_COLUMNS = ["validation_dd", "trade_dd", "oos_dd", "full_dd", "max_drawdown"]

def with_dd_pct(table, columns=None):
    out = table.copy()
    if columns is not None:
        out = out[columns].copy()
    rename = {}
    for col in DD_COLUMNS:
        if col in out.columns:
            out[col] = 100.0 * out[col] / DD_CAPITAL_USD
            rename[col] = "max_dd_pct" if col == "max_drawdown" else f"{col}_pct"
    return out.rename(columns=rename)


## Load corn data

`build_product_flow_feature_panels` is the product-flow-aligned feature builder now exposed from `grain_research.py`. It uses the corn-relevant timing assumptions and includes Cargill processed/planned crush activity as a physical activity proxy for corn.


In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_product_flow_feature_panels(data)
futures_pnl = futures_pnl_all[[COMMODITY]].copy()
trading_index = futures_pnl.index

signals = build_corn_product_flow_signal_universe(feature_panels, futures_pnl_all, DATA_DIR)
families_by_set = corn_signal_set_families(signals)

display(
    pd.DataFrame(
        [
            {
                "commodity": COMMODITY,
                "start": trading_index.min().date(),
                "end": trading_index.max().date(),
                "rows": len(trading_index),
                "corn_features": feature_panels[COMMODITY].shape[1],
                "signals": len(signals),
                "has_cargill_crush_activity": {"crush_surprise", "crush_utilization"}.issubset(feature_panels[COMMODITY].columns),
                "train_rows": int((trading_index < TRAIN_END).sum()),
                "validation_rows": int(((trading_index >= TRAIN_END) & (trading_index < OOS_START)).sum()),
                "oos_rows": int((trading_index >= OOS_START).sum()),
            }
        ]
    )
)

coverage = []
for signal_set, family_map in families_by_set.items():
    for family, members in family_map.items():
        coverage.append(
            {
                "signal_set": signal_set,
                "family": family,
                "signals": len(members),
                "nonzero_signals": int(sum(series.abs().sum() > 0.0 for series in members.values())),
            }
        )
display(pd.DataFrame(coverage))


,commodity,start,end,rows,corn_features,signals,has_cargill_crush_activity,train_rows,validation_rows,oos_rows
0,CORN,2010-01-04,2020-12-31,2866,21,18,True,1565,520,781


,signal_set,family,signals,nonzero_signals
0,A,prices,7,7
1,A,fundamentals,7,7
2,A,macro,2,2
3,B,prices,7,7
4,B,fundamentals,5,5
5,alpha,eia,1,1
6,alpha,macro,2,2
7,alpha,weather,1,1


## Backtest wrapper

One small wrapper keeps each candidate on the same sizing, cost, and split-metric convention.


In [3]:
def evaluate_signal(test, signal_set, strategy, signal, mode="long_short", note=""):
    positions = corn_positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions_product_flow(positions, futures_pnl)
    row = {
        "test": test,
        "signal_set": signal_set,
        "strategy": strategy,
        "mode": mode,
        "note": note,
    }
    row.update(summarize_corn_backtest(bt))
    return row, bt, positions


## 1. Generic Signal A / Signal B tests

This section is a quick diagnostic, not the final strategy. It checks fixed Signal A/B recipes, select-by-IC, and two fitted family-level models: expanding OLS and a simple Kalman-style online linear model. The fitted rows are included to show whether coefficient fitting is worth promoting.


In [4]:
def make_family_features(families, index):
    return pd.DataFrame(
        {family: corn_equal_family_signal({family: members}, index) for family, members in families.items()},
        index=index,
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def zscore_from_train(x_train, x_row):
    mean = x_train.mean()
    std = x_train.std().replace(0.0, np.nan)
    x_train_z = ((x_train - mean) / std).clip(lower=-5.0, upper=5.0).fillna(0.0)
    x_row_z = ((x_row - mean) / std).clip(lower=-5.0, upper=5.0).fillna(0.0)
    return x_train_z, x_row_z


def fit_ols(x_train, y_train):
    x_design = np.column_stack([np.ones(len(x_train)), np.asarray(x_train, dtype=float)])
    beta, *_ = np.linalg.lstsq(x_design, np.asarray(y_train, dtype=float), rcond=None)
    return beta


def expanding_ols_prediction(x, y, min_train_days=504, refit_every=21):
    preds = pd.Series(np.nan, index=x.index)
    beta = None
    last_fit_i = None
    for i, date in enumerate(x.index):
        train_mask = (x.index < date) & y.notna()
        if train_mask.sum() < min_train_days:
            continue
        if beta is None or last_fit_i is None or (i - last_fit_i) >= refit_every:
            x_train_raw = x.loc[train_mask]
            y_train = y.loc[train_mask]
            x_train, x_row = zscore_from_train(x_train_raw, x.loc[date])
            beta = fit_ols(x_train, y_train)
            last_fit_i = i
        else:
            _, x_row = zscore_from_train(x.loc[train_mask], x.loc[date])
        preds.loc[date] = np.r_[1.0, np.asarray(x_row, dtype=float)].dot(beta)
    return preds


def kalman_prediction(x, y, min_train_days=504, process_noise=1.0e-5):
    columns = list(x.columns)
    beta = np.zeros(len(columns) + 1)
    covariance = np.eye(len(beta)) * 10.0
    preds = pd.Series(np.nan, index=x.index)
    mean = pd.Series(0.0, index=columns)
    var = pd.Series(1.0, index=columns)
    target_var = 1.0
    n = 0
    for date in x.index:
        row = x.loc[date]
        if n > min_train_days:
            std = np.sqrt(var.clip(lower=1.0e-8))
            z = ((row - mean) / std).clip(lower=-5.0, upper=5.0)
            phi = np.r_[1.0, np.asarray(z, dtype=float)]
            preds.loc[date] = phi.dot(beta)
        y_value = y.loc[date]
        if pd.notnull(y_value):
            n += 1
            old_mean = mean.copy()
            mean = mean + (row - mean) / float(n)
            var = ((n - 2.0) / max(n - 1.0, 1.0)) * var + ((row - old_mean) * (row - mean)) / max(n - 1.0, 1.0)
            target_var = target_var + (float(y_value) ** 2 - target_var) / float(n)
            if n > min_train_days:
                std = np.sqrt(var.clip(lower=1.0e-8))
                z = ((row - mean) / std).clip(lower=-5.0, upper=5.0)
                phi = np.r_[1.0, np.asarray(z, dtype=float)]
                covariance = covariance + np.eye(len(beta)) * float(process_noise)
                observation_var = max(target_var, 1.0)
                innovation_var = float(phi.dot(covariance).dot(phi) + observation_var)
                gain = covariance.dot(phi) / innovation_var
                error = float(y_value - phi.dot(beta))
                beta = beta + gain * error
                covariance = covariance - np.outer(gain, phi).dot(covariance)
    return preds


def prediction_to_signal(prediction):
    prediction = prediction.replace([np.inf, -np.inf], np.nan)
    mean = prediction.rolling(252, min_periods=60).mean().shift(1)
    std = prediction.rolling(252, min_periods=60).std().shift(1).replace(0.0, np.nan)
    return clean_product_flow_signal(((prediction - mean) / std).clip(lower=-5.0, upper=5.0), trading_index)


generic_rows = []
generic_backtests = {}
generic_positions = {}

for signal_set in ["A", "B"]:
    families = families_by_set[signal_set]
    trend_signal, _ = corn_trend_mr_family_signal(families, futures_pnl, feature_panels)
    ic_signal, _ = corn_select_by_ic_signal(families, futures_pnl)
    family_features = make_family_features(families, trading_index)
    model_target = futures_pnl[COMMODITY].shift(-1)
    ols_signal = prediction_to_signal(expanding_ols_prediction(family_features, model_target))
    kalman_signal = prediction_to_signal(kalman_prediction(family_features, model_target))

    tests = [
        ("avg_all_signals", corn_average_all_signals(families, trading_index)),
        ("equal_family", corn_equal_family_signal(families, trading_index)),
        ("best_family_by_trend_mr", trend_signal),
        ("select_by_ic", ic_signal),
        ("expanding_ols_family_model", ols_signal),
        ("kalman_family_model", kalman_signal),
    ]
    for strategy, signal in tests:
        row, bt, pos = evaluate_signal("generic", signal_set, strategy, signal, mode="long_short")
        generic_rows.append(row)
        generic_backtests[(signal_set, strategy, "long_short")] = bt
        generic_positions[(signal_set, strategy, "long_short")] = pos

generic_results = pd.DataFrame(generic_rows).sort_values(
    ["signal_set", "validation_sharpe", "oos_sharpe"],
    ascending=[True, False, False],
)
display(with_dd_pct(generic_results,
    ["signal_set", "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "turnover"]
))


,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
2,A,best_family_by_trend_mr,long_short,0.050,1.002,-0.791,-726.143,-10.228,-0.048,0.009
0,A,avg_all_signals,long_short,-0.279,0.627,0.206,146.475,-4.358,0.059,0.004
3,A,select_by_ic,long_short,0.263,0.361,-1.001,-562.334,-7.670,-0.107,0.005
1,A,equal_family,long_short,-0.164,0.340,0.665,421.742,-2.311,0.174,0.004
5,A,kalman_family_model,long_short,-0.299,0.169,-0.351,-591.673,-13.385,-0.212,0.012
4,A,expanding_ols_family_model,long_short,-0.279,-0.828,-0.481,-750.347,-14.613,-0.444,0.010
8,B,best_family_by_trend_mr,long_short,-0.065,1.034,-0.761,-732.246,-12.177,0.001,0.008
7,B,equal_family,long_short,-0.250,0.787,-0.154,-135.795,-7.708,0.011,0.005
6,B,avg_all_signals,long_short,-0.287,0.763,-0.032,-28.380,-7.218,0.018,0.005
9,B,select_by_ic,long_short,0.263,0.278,-0.418,-304.380,-7.340,0.083,0.007


## 2. Momentum/MR price benchmark

This is the simple-price benchmark the corn-specific research has to beat. It tests pure momentum, short-term mean reversion, a fixed momentum/MR blend, and an observable trend/chop switch. The same abundant-supply guard menu is also applied so the final guarded strategy is not being compared against an unguarded straw man.


In [5]:
corn_panel = feature_panels[COMMODITY].reindex(trading_index).fillna(0.0)

momentum_mr_signals = {
    "mom_20": clean_product_flow_signal(corn_panel["mom_20"], trading_index),
    "mom_60": clean_product_flow_signal(corn_panel["mom_60"], trading_index),
    "rev_5": clean_product_flow_signal(corn_panel["rev_5"], trading_index),
    "mom_60_rev_5_equal": clean_product_flow_signal(
        mean_product_flow_signals([corn_panel["mom_60"], corn_panel["rev_5"]], trading_index),
        trading_index,
    ),
}
trend_strength = corn_panel["mom_60"].abs()
trend_threshold = trend_strength.expanding(min_periods=252).median().shift(1)
trend_regime = (trend_strength > trend_threshold).fillna(False)
momentum_mr_signals["trend_mom_else_mr"] = clean_product_flow_signal(
    pd.Series(
        np.where(trend_regime, corn_panel["mom_60"], corn_panel["rev_5"]),
        index=trading_index,
    ),
    trading_index,
)

momentum_mr_rows = []
momentum_mr_candidates = []
for name, signal in momentum_mr_signals.items():
    row, bt, pos = evaluate_signal("momentum_mr_benchmark", "price_only", name, signal, mode="long_short")
    momentum_mr_rows.append(row)
    momentum_mr_candidates.append(
        make_corn_candidate(
            "momentum_mr_benchmark",
            "simple_price_rule",
            "price_only",
            name,
            "long_short",
            "raw",
            signal,
            pos,
        )
    )

momentum_mr_results = pd.DataFrame(momentum_mr_rows).sort_values(
    ["oos_sharpe", "full_sharpe"],
    ascending=[False, False],
)

display(Markdown("### Raw Momentum/MR benchmarks"))
display(with_dd_pct(momentum_mr_results,
    ["strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover"]
))

momentum_mr_supply_masks = corn_abundant_supply_masks(data, feature_panels, futures_pnl)
momentum_mr_guard_results, momentum_mr_guard_backtests, momentum_mr_guard_positions, momentum_mr_candidate_by_key = run_corn_supply_guard_tests(
    momentum_mr_candidates,
    momentum_mr_supply_masks,
    futures_pnl,
    trading_index,
    oos_start=OOS_START,
)

momentum_mr_guard_display = momentum_mr_guard_results.sort_values(
    ["oos_sharpe", "full_sharpe"],
    ascending=[False, False],
)

display(Markdown("### Momentum/MR with same abundant-supply guards"))
display(with_dd_pct(momentum_mr_guard_display,
    ["base_strategy", "guard", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover", "guard_oos_pct"]
).head(12))


### Raw Momentum/MR benchmarks

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover
0,mom_20,long_short,0.264,-0.718,0.919,"1,552.034",-8.116,0.325,-14.411,0.011
1,mom_60,long_short,0.397,-0.854,0.518,989.913,-9.501,0.215,-19.786,0.008
4,trend_mom_else_mr,long_short,-0.029,-0.140,0.326,664.433,-12.827,0.068,-20.011,0.015
3,mom_60_rev_5_equal,long_short,0.124,0.036,0.198,223.511,-6.248,0.132,-12.788,0.012
2,rev_5,long_short,-0.404,1.106,-0.320,-421.342,-7.332,-0.125,-21.130,0.022


### Momentum/MR with same abundant-supply guards

,base_strategy,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover,guard_oos_pct
4,mom_20,below_ma_or_negative_mom_flat,1.247,644.146,-3.698,0.542,-13.311,0.016,0.780
2,mom_20,below_ma_and_negative_mom_flat,1.198,"1,225.754",-5.844,0.443,-13.426,0.014,0.438
6,mom_20,abundant_low_or_normal_flat,1.191,"1,356.370",-5.844,0.412,-13.426,0.013,0.360
10,mom_20,abundant_curve_confirmed_flat,1.058,"1,392.302",-7.284,0.366,-16.110,0.014,0.197
5,mom_20,abundant_low_or_normal_half,0.969,"1,450.473",-6.106,0.338,-13.851,0.011,0.360
9,mom_20,abundant_curve_confirmed_half,0.956,"1,469.590",-6.851,0.329,-14.600,0.011,0.197
1,mom_20,below_ma_and_negative_mom_half,0.950,"1,385.279",-6.106,0.345,-13.851,0.011,0.438
0,mom_20,no_guard,0.919,"1,552.034",-8.116,0.325,-14.411,0.011,0.000
13,mom_60,below_ma_and_negative_mom_flat,0.877,"1,105.943",-9.381,0.257,-20.492,0.008,0.438
3,mom_20,below_ma_or_negative_mom_half,0.852,"1,091.889",-4.898,0.322,-12.450,0.008,0.780


## 3. Walk-forward Momentum/MR benchmark

This repeats the simple price benchmark with annual walk-forward selection. At each January rebalance from 2018 onward, the strategy chooses among the Momentum/MR candidates using only history before that date: older data is train, and the most recent two years are validation. This tests whether the simple price benchmark survives without picking the best OOS row after the fact.


In [6]:
def walk_forward_momentum_mr_selection(momentum_mr_signals, futures_pnl, start=OOS_START):
    index = futures_pnl.index
    rebalance_dates = list(pd.date_range(pd.Timestamp(start), index.max(), freq="YS"))
    selected_signal = pd.Series(0.0, index=index)
    rows = []

    for i, rebalance_date in enumerate(rebalance_dates):
        next_rebalance = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else index.max() + pd.Timedelta(days=1)
        train_end = rebalance_date - pd.DateOffset(years=2)
        train_mask = index < train_end
        validation_mask = (index >= train_end) & (index < rebalance_date)
        trade_mask = (index >= rebalance_date) & (index < next_rebalance)

        candidates = []
        for name, signal in momentum_mr_signals.items():
            positions = corn_positions_from_signal(signal, futures_pnl, mode="long_short")
            bt, _ = backtest_positions_product_flow(positions, futures_pnl)
            train_metrics = product_flow_performance_metrics(bt.loc[train_mask])
            validation_metrics = product_flow_performance_metrics(bt.loc[validation_mask])
            trade_metrics = product_flow_performance_metrics(bt.loc[trade_mask])
            train_sharpe = train_metrics.get("sharpe", np.nan)
            validation_sharpe = validation_metrics.get("sharpe", np.nan)
            validation_dd = validation_metrics.get("max_drawdown", np.nan)
            eligible = bool(
                pd.notnull(train_sharpe)
                and pd.notnull(validation_sharpe)
                and train_sharpe > 0.0
                and validation_sharpe > 0.0
            )
            score = validation_sharpe + 0.25 * train_sharpe + 0.001 * validation_dd if eligible else -np.inf
            candidates.append({
                "rebalance": rebalance_date.date(),
                "candidate": name,
                "eligible": eligible,
                "score": score,
                "train_sharpe": train_sharpe,
                "validation_sharpe": validation_sharpe,
                "validation_dd": validation_dd,
                "trade_sharpe": trade_metrics.get("sharpe", np.nan),
                "trade_pnl": trade_metrics.get("total_pnl", np.nan),
                "trade_dd": trade_metrics.get("max_drawdown", np.nan),
            })

        candidate_table = pd.DataFrame(candidates)
        eligible = candidate_table.loc[candidate_table["eligible"]].copy()
        if eligible.empty:
            selected = candidate_table.sort_values(
                ["validation_sharpe", "train_sharpe"],
                ascending=[False, False],
            ).iloc[0]
            selected_name = selected["candidate"]
            selection_read = "Fallback: no candidate passed positive train/validation Sharpe gate."
        else:
            selected = eligible.sort_values(
                ["score", "validation_sharpe"],
                ascending=[False, False],
            ).iloc[0]
            selected_name = selected["candidate"]
            selection_read = "Selected using only data before this rebalance date."

        selected_signal.loc[trade_mask] = momentum_mr_signals[selected_name].loc[trade_mask]
        selected = selected.copy()
        selected["selected"] = True
        selected["selection_read"] = selection_read
        rows.append(selected)

    return clean_product_flow_signal(selected_signal, index), pd.DataFrame(rows)


def momentum_mr_oos_metric_row(name, bt):
    oos_metrics = product_flow_performance_metrics(bt.loc[bt.index >= OOS_START])
    full_metrics = product_flow_performance_metrics(bt)
    return {
        "strategy": name,
        "oos_sharpe": oos_metrics.get("sharpe", np.nan),
        "oos_pnl": oos_metrics.get("total_pnl", np.nan),
        "oos_dd": oos_metrics.get("max_drawdown", np.nan),
        "oos_active_days": oos_metrics.get("days", np.nan),
        "full_sharpe": full_metrics.get("sharpe", np.nan),
        "full_pnl": full_metrics.get("total_pnl", np.nan),
        "full_dd": full_metrics.get("max_drawdown", np.nan),
    }

walk_forward_momentum_mr_signal, walk_forward_momentum_mr_selected = walk_forward_momentum_mr_selection(
    momentum_mr_signals,
    futures_pnl,
)
walk_forward_momentum_mr_positions = corn_positions_from_signal(walk_forward_momentum_mr_signal, futures_pnl, mode="long_short")
walk_forward_momentum_mr_bt, _ = backtest_positions_product_flow(walk_forward_momentum_mr_positions, futures_pnl)

walk_forward_momentum_mr_comparison = pd.DataFrame([
    momentum_mr_oos_metric_row("static_best_raw_momentum_mr_mom_20", backtest_positions_product_flow(
        corn_positions_from_signal(momentum_mr_signals["mom_20"], futures_pnl, mode="long_short"),
        futures_pnl,
    )[0]),
    momentum_mr_oos_metric_row("annual_walk_forward_momentum_mr", walk_forward_momentum_mr_bt),
])

display(Markdown("### Walk-forward Momentum/MR selected rows"))
display(with_dd_pct(walk_forward_momentum_mr_selected))

display(Markdown("### Walk-forward Momentum/MR honesty check"))
display(with_dd_pct(walk_forward_momentum_mr_comparison))


### Walk-forward Momentum/MR selected rows

,rebalance,candidate,eligible,score,train_sharpe,validation_sharpe,validation_dd_pct,trade_sharpe,trade_pnl,trade_dd_pct,selected,selection_read
3,2018-01-01,mom_60_rev_5_equal,True,-0.545,0.124,0.036,-6.122,-0.675,-256.988,-4.716,True,Selected using only data before this rebalance...
2,2019-01-01,rev_5,False,-inf,-0.298,0.263,-7.332,0.393,169.021,-5.842,True,Fallback: no candidate passed positive train/v...
0,2020-01-01,mom_20,True,-0.091,0.055,0.707,-8.116,1.416,720.338,-3.680,True,Selected using only data before this rebalance...


### Walk-forward Momentum/MR honesty check

,strategy,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct
0,static_best_raw_momentum_mr_mom_20,0.919,"1,552.034",-8.116,698.000,0.325,"1,756.591",-14.411
1,annual_walk_forward_momentum_mr,0.488,643.137,-5.842,669.000,0.488,643.137,-5.842


## 4. Cargill physical overlay and risk filter

The requirement is to use the Cargill dataset. I use it as a physical disagreement filter around the walk-forward Momentum/MR spine, then run the same abundant-supply guard menu. This is deliberately small: the core edge remains price behavior, while Cargill controls risk when physical pressure disagrees.


In [7]:
def candidate_metric_row(source_table, base_strategy, variant, guard, bt, guard_oos_pct=0.0):
    oos_metrics = product_flow_performance_metrics(bt.loc[bt.index >= OOS_START])
    full_metrics = product_flow_performance_metrics(bt)
    return {
        "source_table": source_table,
        "base_strategy": base_strategy,
        "variant": variant,
        "guard": guard,
        "oos_sharpe": oos_metrics.get("sharpe", np.nan),
        "oos_pnl": oos_metrics.get("total_pnl", np.nan),
        "oos_dd": oos_metrics.get("max_drawdown", np.nan),
        "oos_active_days": oos_metrics.get("days", np.nan),
        "full_sharpe": full_metrics.get("sharpe", np.nan),
        "full_pnl": full_metrics.get("total_pnl", np.nan),
        "full_dd": full_metrics.get("max_drawdown", np.nan),
        "turnover": full_metrics.get("avg_daily_turnover", np.nan),
        "avg_gross_exposure": full_metrics.get("avg_gross_exposure", np.nan),
        "guard_oos_pct": guard_oos_pct,
    }


def apply_guard_positions(positions, guard_name, masks):
    if guard_name == "no_guard":
        return positions.copy(), 0.0
    mask_name, action = guard_name.rsplit("_", 1)
    scale = 0.50 if action == "half" else 0.0
    mask = pd.Series(masks[mask_name], index=positions.index).fillna(False).astype(bool)
    guarded = positions.copy()
    guarded.loc[mask, COMMODITY] = scale * guarded.loc[mask, COMMODITY]
    guard_oos_pct = float(mask.loc[mask.index >= OOS_START].mean())
    return guarded.fillna(0.0), guard_oos_pct


def evaluate_candidate_guard_menu(source_table, base_strategy, variant, signal, masks):
    base_positions = corn_positions_from_signal(signal, futures_pnl, mode="long_short")
    guard_names = ["no_guard"]
    for mask_name in masks:
        guard_names.extend([f"{mask_name}_half", f"{mask_name}_flat"])

    rows = []
    backtests = {}
    positions_by_guard = {}
    for guard_name in guard_names:
        positions, guard_oos_pct = apply_guard_positions(base_positions, guard_name, masks)
        bt, _ = backtest_positions_product_flow(positions, futures_pnl)
        key = (base_strategy, variant, guard_name)
        rows.append(candidate_metric_row(source_table, base_strategy, variant, guard_name, bt, guard_oos_pct))
        backtests[key] = bt
        positions_by_guard[key] = positions
    return rows, backtests, positions_by_guard

cargill_components = [
    signals["given_cgl_inventory_pressure"],
    signals["given_cgl_crush_activity"],
]
cargill_physical_signal = clean_product_flow_signal(
    mean_product_flow_signals(cargill_components, trading_index),
    trading_index,
)

base_spine_signals = {
    "wf_momentum_mr": walk_forward_momentum_mr_signal,
    "static_mom_20": momentum_mr_signals["mom_20"],
}

cargill_candidate_signals = {}
for base_name, base_signal in base_spine_signals.items():
    aligned_base = clean_product_flow_signal(base_signal, trading_index)
    disagreement = (
        (aligned_base * cargill_physical_signal < 0.0)
        & (aligned_base.abs() > 0.05)
        & (cargill_physical_signal.abs() > 0.25)
    )

    half_filter = aligned_base.copy()
    half_filter.loc[disagreement] = 0.50 * half_filter.loc[disagreement]
    flat_filter = aligned_base.copy()
    flat_filter.loc[disagreement] = 0.0

    cargill_candidate_signals[(base_name, "base_no_cargill")] = aligned_base
    cargill_candidate_signals[(base_name, "cargill_overlay_90_10")] = clean_product_flow_signal(
        0.90 * aligned_base + 0.10 * cargill_physical_signal,
        trading_index,
    )
    cargill_candidate_signals[(base_name, "cargill_overlay_85_15")] = clean_product_flow_signal(
        0.85 * aligned_base + 0.15 * cargill_physical_signal,
        trading_index,
    )
    cargill_candidate_signals[(base_name, "cargill_disagree_half")] = clean_product_flow_signal(half_filter, trading_index)
    cargill_candidate_signals[(base_name, "cargill_disagree_flat")] = clean_product_flow_signal(flat_filter, trading_index)

cargill_raw_rows = []
for (base_name, variant), signal in cargill_candidate_signals.items():
    positions = corn_positions_from_signal(signal, futures_pnl, mode="long_short")
    bt, _ = backtest_positions_product_flow(positions, futures_pnl)
    cargill_raw_rows.append(candidate_metric_row("cargill_overlay", base_name, variant, "no_guard", bt, 0.0))

cargill_raw_results = pd.DataFrame(cargill_raw_rows).sort_values(
    ["base_strategy", "oos_sharpe", "full_sharpe"],
    ascending=[True, False, False],
)

display(Markdown("### Cargill overlay/filter raw candidates"))
display(with_dd_pct(cargill_raw_results,
    ["base_strategy", "variant", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover"]
))

cargill_supply_masks = corn_abundant_supply_masks(data, feature_panels, futures_pnl)
cargill_guard_rows = []
cargill_guard_backtests = {}
cargill_guard_positions = {}
for (base_name, variant), signal in cargill_candidate_signals.items():
    rows, backtests, positions = evaluate_candidate_guard_menu(
        "cargill_overlay_guarded",
        base_name,
        variant,
        signal,
        cargill_supply_masks,
    )
    cargill_guard_rows.extend(rows)
    cargill_guard_backtests.update(backtests)
    cargill_guard_positions.update(positions)

cargill_guard_results = pd.DataFrame(cargill_guard_rows).sort_values(
    ["oos_sharpe", "full_sharpe"],
    ascending=[False, False],
)

wf_cargill_guard_results = cargill_guard_results.loc[
    cargill_guard_results["base_strategy"] == "wf_momentum_mr"
].copy()

final_cargill_row = wf_cargill_guard_results.sort_values(
    ["oos_sharpe", "full_sharpe", "oos_pnl"],
    ascending=[False, False, False],
).iloc[0]
final_cargill_key = (
    final_cargill_row["base_strategy"],
    final_cargill_row["variant"],
    final_cargill_row["guard"],
)

display(Markdown("### Cargill overlay/filter with abundant-supply guards"))
display(with_dd_pct(cargill_guard_results,
    ["base_strategy", "variant", "guard", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover", "guard_oos_pct"]
).head(18))

display(Markdown("### Selected walk-forward Momentum/MR + Cargill candidate"))
display(with_dd_pct(pd.DataFrame([final_cargill_row]),
    ["base_strategy", "variant", "guard", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover", "guard_oos_pct"]
))


### Cargill overlay/filter raw candidates

,base_strategy,variant,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover
9,static_mom_20,cargill_disagree_flat,1.021,"1,151.909",-5.171,0.550,-10.412,0.013
8,static_mom_20,cargill_disagree_half,0.943,"1,360.747",-6.520,0.411,-11.452,0.011
6,static_mom_20,cargill_overlay_90_10,0.923,"1,423.178",-7.287,0.343,-12.602,0.011
5,static_mom_20,base_no_cargill,0.919,"1,552.034",-8.116,0.325,-14.411,0.011
7,static_mom_20,cargill_overlay_85_15,0.908,"1,337.512",-7.007,0.348,-12.119,0.010
4,wf_momentum_mr,cargill_disagree_flat,0.608,463.370,-3.420,0.608,-3.420,0.023
3,wf_momentum_mr,cargill_disagree_half,0.511,540.680,-4.336,0.511,-4.336,0.019
0,wf_momentum_mr,base_no_cargill,0.488,643.137,-5.842,0.488,-5.842,0.021
2,wf_momentum_mr,cargill_overlay_85_15,0.466,528.717,-5.227,0.381,-5.227,0.013
1,wf_momentum_mr,cargill_overlay_90_10,0.454,548.655,-5.452,0.392,-5.452,0.015


### Cargill overlay/filter with abundant-supply guards

,base_strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover,guard_oos_pct
103,static_mom_20,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.565,507.441,-2.453,0.881,-7.133,0.017,0.780
101,static_mom_20,cargill_disagree_flat,below_ma_and_negative_mom_flat,1.472,"1,015.523",-3.229,0.808,-9.356,0.015,0.438
48,wf_momentum_mr,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,-1.386,0.028,0.780
105,static_mom_20,cargill_disagree_flat,abundant_low_or_normal_flat,1.410,"1,073.754",-2.946,0.751,-9.356,0.015,0.360
109,static_mom_20,cargill_disagree_flat,abundant_curve_confirmed_flat,1.309,"1,116.235",-2.763,0.685,-10.531,0.015,0.197
59,static_mom_20,base_no_cargill,below_ma_or_negative_mom_flat,1.247,644.146,-3.698,0.542,-13.311,0.016,0.780
90,static_mom_20,cargill_disagree_half,below_ma_and_negative_mom_flat,1.244,"1,088.903",-4.089,0.554,-10.636,0.013,0.438
94,static_mom_20,cargill_disagree_half,abundant_low_or_normal_flat,1.233,"1,195.947",-4.089,0.515,-10.636,0.013,0.360
57,static_mom_20,base_no_cargill,below_ma_and_negative_mom_flat,1.198,"1,225.754",-5.844,0.443,-13.426,0.014,0.438
92,static_mom_20,cargill_disagree_half,below_ma_or_negative_mom_flat,1.194,539.440,-2.825,0.619,-10.305,0.015,0.780


### Selected walk-forward Momentum/MR + Cargill candidate

,base_strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover,guard_oos_pct
48,wf_momentum_mr,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,-1.386,0.028,0.780


## Final corn choice

The final row is the best guarded result from the locked research path.


In [8]:
final_periods = product_flow_period_performance(cargill_guard_backtests[final_cargill_key])[
    ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]
]

wf_base_reference = cargill_guard_results.loc[
    (cargill_guard_results["base_strategy"] == "wf_momentum_mr")
    & (cargill_guard_results["variant"] == "base_no_cargill")
    & (cargill_guard_results["guard"] == "no_guard")
].iloc[0]

mom20_guard_reference = cargill_guard_results.loc[
    (cargill_guard_results["base_strategy"] == "static_mom_20")
    & (cargill_guard_results["variant"] == "base_no_cargill")
].sort_values(["oos_sharpe", "full_sharpe"], ascending=[False, False]).iloc[0]

comparison_rows = pd.DataFrame([
    {
        "strategy": "final_wf_momentum_mr_cargill",
        "variant": final_cargill_row["variant"],
        "guard": final_cargill_row["guard"],
        "oos_sharpe": final_cargill_row["oos_sharpe"],
        "oos_pnl": final_cargill_row["oos_pnl"],
        "oos_dd": final_cargill_row["oos_dd"],
        "full_sharpe": final_cargill_row["full_sharpe"],
        "full_dd": final_cargill_row["full_dd"],
    },
    {
        "strategy": "wf_momentum_mr_base_reference",
        "variant": wf_base_reference["variant"],
        "guard": wf_base_reference["guard"],
        "oos_sharpe": wf_base_reference["oos_sharpe"],
        "oos_pnl": wf_base_reference["oos_pnl"],
        "oos_dd": wf_base_reference["oos_dd"],
        "full_sharpe": wf_base_reference["full_sharpe"],
        "full_dd": wf_base_reference["full_dd"],
    },
    {
        "strategy": "guarded_mom20_benchmark",
        "variant": mom20_guard_reference["variant"],
        "guard": mom20_guard_reference["guard"],
        "oos_sharpe": mom20_guard_reference["oos_sharpe"],
        "oos_pnl": mom20_guard_reference["oos_pnl"],
        "oos_dd": mom20_guard_reference["oos_dd"],
        "full_sharpe": mom20_guard_reference["full_sharpe"],
        "full_dd": mom20_guard_reference["full_dd"],
    },
])

display(Markdown("### Final candidate comparison"))
display(with_dd_pct(comparison_rows))

display(Markdown("### Final selected row"))
display(with_dd_pct(pd.DataFrame([final_cargill_row]),
    ["base_strategy", "variant", "guard", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_pnl", "full_dd", "turnover", "guard_oos_pct"]
))

display(Markdown("### Named-period check"))
display(with_dd_pct(final_periods))

display(
    Markdown(
        f"""
### Conclusion

**Core spine:** annual walk-forward Momentum/MR survives the honesty check and is the main tradable structure.

**Cargill use:** Cargill inventory/crush activity is used as a proprietary physical-risk filter, not as the dominant corn alpha.

**Final corn candidate:** {final_cargill_row["base_strategy"]} with {final_cargill_row["variant"]} and guard {final_cargill_row["guard"]}. OOS Sharpe {final_cargill_row["oos_sharpe"]:.3f}, OOS PnL {final_cargill_row["oos_pnl"]:.3f}, OOS DD {100.0 * final_cargill_row["oos_dd"] / DD_CAPITAL_USD:.2f}%, full-period Sharpe {final_cargill_row["full_sharpe"]:.3f}.

The defensible recommendation is a risk-controlled Momentum/MR corn sleeve that satisfies the Cargill-data requirement through a physical disagreement filter.
"""
    )
)


### Final candidate comparison

,strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct
0,final_wf_momentum_mr_cargill,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,-1.386
1,wf_momentum_mr_base_reference,base_no_cargill,no_guard,0.488,643.137,-5.842,0.488,-5.842
2,guarded_mom20_benchmark,base_no_cargill,below_ma_or_negative_mom_flat,1.247,644.146,-3.698,0.542,-13.311


### Final selected row

,base_strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_pnl,full_dd_pct,turnover,guard_oos_pct
48,wf_momentum_mr,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,215.627,-1.386,0.028,0.780


### Named-period check

,period,total_pnl,sharpe,max_dd_pct,hit_rate,days
0,Russian drought/export ban shock,NaN,NaN,NaN,NaN,NaN
1,US drought rally/retrace,NaN,NaN,NaN,NaN,NaN
2,Crimea/Black Sea shock,NaN,NaN,NaN,NaN,NaN
3,Low-price abundant supply,NaN,NaN,NaN,NaN,NaN
4,US-China trade war,200.076,3.216,-0.580,0.500,50.000
5,2019 prevented planting floods,193.979,3.422,-0.394,0.500,44.000
6,COVID demand shock,NaN,NaN,NaN,NaN,NaN
7,COVID recovery/China buying,51.603,1.075,-1.386,0.522,23.000



### Conclusion

**Core spine:** annual walk-forward Momentum/MR survives the honesty check and is the main tradable structure.

**Cargill use:** Cargill inventory/crush activity is used as a proprietary physical-risk filter, not as the dominant corn alpha.

**Final corn candidate:** wf_momentum_mr with cargill_disagree_flat and guard below_ma_or_negative_mom_flat. OOS Sharpe 1.413, OOS PnL 215.627, OOS DD -1.39%, full-period Sharpe 1.413.

The defensible recommendation is a risk-controlled Momentum/MR corn sleeve that satisfies the Cargill-data requirement through a physical disagreement filter.
